# Image Captioning

Image captioning is a **cross-modal generation** task: given an image, produce a natural language sentence that describes its content. This requires a model to simultaneously understand visual structure and generate fluent text — bridging computer vision and natural language processing in a single architecture. Early neural approaches used a CNN encoder to extract a fixed-length image feature vector, which was then decoded by an RNN. A key improvement was **soft attention** [@showattendtell], which lets the decoder selectively focus on different regions of the image at each generation step rather than relying on a single global summary.

We adopt the modern Transformer-based formulation. The **encoder** is a Vision Transformer (ViT) [@vit] that divides the image into a grid of non-overlapping patches, linearly projects each patch to an embedding vector, and processes the resulting sequence with multi-head self-attention blocks to produce $N$ contextualized patch representations. The **decoder** is a stack of Transformer decoder blocks — identical to those developed in the [machine translation notebook](./13-translation.html) — each of which applies causal self-attention over the caption prefix, followed by cross-attention over the $N$ patch tokens. This cross-attention is the mechanism through which the decoder grounds each generated word in specific image regions. The encoder's self-attention and positional encoding are reviewed in [NB09](../09-attention-transformers.html); the decoder block and cross-attention are developed in detail in [NB13](./13-translation.html).

In this notebook we build a complete image captioning pipeline from scratch. We train a ViT patch encoder (no pretrained weights) together with a Transformer decoder on the Flickr8k dataset [@flickr8k] using teacher-forcing. After training we evaluate with greedy decoding, measure translation quality via corpus BLEU-4 [@bleu], and visualize the cross-attention maps to see which patches the decoder attends to when generating each word.

<br>

In [ ]:
import math, re, json, random, warnings
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from matplotlib_inline import backend_inline

DATASET_DIR = Path("./data").resolve()
DATASET_DIR.mkdir(exist_ok=True)

RANDOM_SEED = 0
DEBUG = False
MATPLOTLIB_FORMAT = "png" if DEBUG else "svg"

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
warnings.filterwarnings("ignore")
backend_inline.set_matplotlib_formats(MATPLOTLIB_FORMAT)

DEVICE = (
    torch.device("cuda") if torch.cuda.is_available()
    else torch.device("mps") if torch.backends.mps.is_available()
    else torch.device("cpu")
)
print("device:", DEVICE)

## Dataset

**Data.** Flickr8k [@flickr8k] contains 8,000 images each paired with five human-written captions, giving 40,000 (image, caption) pairs in total. We download it via the Hugging Face `datasets` library. Each example has an `"image"` field (PIL image) and a `"caption"` field (string). We split the 40,000 pairs 80/10/10 into train, validation, and test sets.

In [ ]:
from datasets import load_dataset

raw = load_dataset("jxie/flickr8k")  # <1>
all_examples = list(raw["train"])    # <2>

random.seed(RANDOM_SEED)
random.shuffle(all_examples)

N = len(all_examples)
n_train = int(0.8 * N)
n_val   = int(0.1 * N)

train_examples = all_examples[:n_train]
val_examples   = all_examples[n_train:n_train + n_val]
test_examples  = all_examples[n_train + n_val:]

print(f"Total: {N} | Train: {len(train_examples)} | Val: {len(val_examples)} | Test: {len(test_examples)}")

1. The `jxie/flickr8k` dataset on Hugging Face contains all 40,000 (image, caption) pairs under a single `train` split.
2. We convert to a list so we can shuffle and slice in-place.

**Vocabulary.** We build a word-level vocabulary from the training captions. Tokenization lowercases each caption, strips punctuation, and splits on whitespace. Words appearing fewer than 5 times are mapped to `<unk>`. Four special tokens are reserved: `<pad>` at index 0, `<bos>` at index 1, `<eos>` at index 2, and `<unk>` at index 3.

In [ ]:
from collections import Counter

PAD_IDX, BOS_IDX, EOS_IDX, UNK_IDX = 0, 1, 2, 3
MIN_FREQ   = 5
MAX_CAP_LEN = 40

def tokenize(caption: str):
    caption = caption.lower()
    caption = re.sub(r"[^\w\s]", "", caption)
    return caption.split()

# count word frequencies on training set
counter = Counter()
for ex in train_examples:
    counter.update(tokenize(ex["caption"]))

# build vocabulary: special tokens first, then words by frequency
special_tokens = ["<pad>", "<bos>", "<eos>", "<unk>"]
vocab_words = [w for w, c in counter.most_common() if c >= MIN_FREQ]
itos = special_tokens + vocab_words
stoi = {w: i for i, w in enumerate(itos)}

VOCAB_SIZE = len(itos)
print(f"Vocabulary size: {VOCAB_SIZE}")
print(f"Train captions: {len(train_examples)} | Val: {len(val_examples)} | Test: {len(test_examples)}")

**CaptionDataset.** Each `__getitem__` returns an image tensor and a caption tensor that includes the `<bos>` and `<eos>` tokens. We define a `collate_fn` that pads captions to the maximum length in each batch.

In [ ]:
IMG_SIZE   = 224
BATCH_SIZE = 64

train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],  # ImageNet stats
                         std =[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])


class CaptionDataset(Dataset):

    def __init__(self, examples, img_transform, max_len=MAX_CAP_LEN):
        self.examples = examples
        self.transform = img_transform
        self.max_len = max_len

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):                           # <1>
        ex = self.examples[idx]
        img = ex["image"].convert("RGB")
        img_tensor = self.transform(img)

        tokens = tokenize(ex["caption"])[:self.max_len]
        ids = [BOS_IDX] + [stoi.get(t, UNK_IDX) for t in tokens] + [EOS_IDX]
        cap_tensor = torch.tensor(ids, dtype=torch.long)
        return img_tensor, cap_tensor


def collate_fn(batch):                                    # <2>
    imgs, caps = zip(*batch)
    imgs = torch.stack(imgs)
    max_len = max(c.size(0) for c in caps)
    padded = torch.full((len(caps), max_len), PAD_IDX, dtype=torch.long)
    for i, c in enumerate(caps):
        padded[i, :c.size(0)] = c
    return imgs, padded


train_dataset = CaptionDataset(train_examples, train_transform)
val_dataset   = CaptionDataset(val_examples,   eval_transform)
test_dataset  = CaptionDataset(test_examples,  eval_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate_fn, num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collate_fn, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collate_fn, num_workers=0)

print(f"Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}")

1. Each item returns a normalized image tensor of shape $(3, 224, 224)$ and a caption tensor of shape $(T+2,)$ — the $T$ caption tokens sandwiched between `<bos>` and `<eos>`, truncated to `MAX_CAP_LEN` content tokens.
2. `collate_fn` pads each batch's captions to the length of the longest caption in that batch, minimizing unnecessary padding.

Displaying four sample images from the training set with their captions:

In [ ]:
#| code-fold: true
#| label: fig-samples
#| fig-cap: "Four sample images from the Flickr8k training set with their associated captions."

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for i, ax in enumerate(axes):
    ex = train_examples[i]
    ax.imshow(ex["image"])
    ax.axis("off")
    caption = ex["caption"]
    words = caption.split()
    # wrap at ~8 words per line
    lines = []
    for j in range(0, min(len(words), 16), 8):
        lines.append(" ".join(words[j:j+8]))
    ax.set_title("\n".join(lines), fontsize=7, wrap=True)
fig.tight_layout()
plt.show()

**Figure.** Sample images from the Flickr8k training set. Each image has five human-written captions; we show one. The captions range from simple object descriptions to action-oriented sentences, capturing the diversity the model must learn to generate.

## Patch embedding

**Patch embedding.** A ViT [@vit] treats the image as a sequence of non-overlapping patches. For a $H \times W$ image divided into $P \times P$ patches, there are $N = (H/P)^2$ patches (assuming $H = W$). Each patch is a vector of size $P^2 \cdot C$ and is projected to $d_\text{model}$ via a linear layer. Concretely, for patch $p_i \in \mathbb{R}^{P^2 C}$, the embedding is

$$\mathbf{e}_i = p_i \mathbf{W}_E + \mathbf{b}_E, \quad \mathbf{W}_E \in \mathbb{R}^{P^2 C \times d_\text{model}}.$$

A learnable `[CLS]` token is prepended, and learnable position embeddings are added to all $N + 1$ positions to indicate spatial location. The result is a sequence of $N + 1$ tokens: `[CLS, patch_1, ..., patch_N]`.

The projection can be implemented efficiently with a single `Conv2d` whose kernel and stride both equal $P$, tiling the image with non-overlapping patches and projecting each one in a single pass.

Defining the `PatchEmbedding` module:

In [ ]:
class PatchEmbedding(nn.Module):

    def __init__(self, img_size=224, patch_size=16, in_channels=3, d_model=256):
        super().__init__()
        self.patch_size = patch_size
        self.n_patches = (img_size // patch_size) ** 2                              # <1>
        self.proj = nn.Conv2d(in_channels, d_model,
                              kernel_size=patch_size, stride=patch_size)            # <2>
        self.cls_token  = nn.Parameter(torch.zeros(1, 1, d_model))                 # <3>
        self.pos_embed  = nn.Parameter(torch.zeros(1, self.n_patches + 1, d_model))# <4>
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)

    def forward(self, x):
        B = x.shape[0]
        x = self.proj(x)                                  # (B, d_model, H/P, W/P)  # <5>
        x = x.flatten(2).transpose(1, 2)                  # (B, N, d_model)          # <6>
        cls = self.cls_token.expand(B, -1, -1)            # (B, 1, d_model)
        x = torch.cat([cls, x], dim=1)                    # (B, N+1, d_model)        # <7>
        x = x + self.pos_embed
        return x

1. For `img_size=224, patch_size=16`: $N = (224/16)^2 = 196$ patches.
2. A `Conv2d` with `kernel_size=patch_size, stride=patch_size` tiles the image with non-overlapping patches and projects each to `d_model` in one step — equivalent to flattening each $P \times P$ patch and applying a shared linear projection.
3. Learnable `[CLS]` token, initialized near zero. After the encoder its representation aggregates global image information.
4. Learnable position embeddings for all $N + 1$ positions (including `[CLS]`), initialized with truncated normal ($\sigma = 0.02$).
5. Output shape: `(B, d_model, H/P, W/P)` — one channel slice per patch position.
6. Flatten spatial dims and transpose: `(B, d_model, N) → (B, N, d_model)`.
7. Prepend `[CLS]` token; sequence length becomes $N + 1 = 197$.

## ViT encoder

**ViT encoder.** The patch embeddings are passed through $L$ encoder blocks — identical to the Pre-LN encoder blocks from [NB09](../09-attention-transformers.html) [@preln-transformer]. Each block applies multi-head self-attention followed by a position-wise FFN, both with Pre-LN residual connections [@transformers]. The output is a sequence of $N + 1$ contextualized patch representations.

For caption generation, we use **all $N$ patch tokens** (excluding `[CLS]`) as the memory sequence that the caption decoder will cross-attend to, giving it spatial coverage of the entire image. The `[CLS]` token aggregates global information and is useful for classification tasks, but here we want the decoder to attend to individual patch locations.

Re-implementing `MultiHeadAttention` and `EncoderBlock` for self-containedness (adapted from NB09/NB13):

In [ ]:
# adapted from NB09/NB13
class MultiHeadAttention(nn.Module):

    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.0):
        super().__init__()
        assert d_model % num_heads == 0, "num_heads must divide d_model"
        self.d_model   = d_model
        self.n_heads   = num_heads
        self.d_head    = d_model // num_heads
        self.dropout_p = dropout
        self.w_q = nn.Linear(d_model, d_model, bias=False)
        self.w_k = nn.Linear(d_model, d_model, bias=False)
        self.w_v = nn.Linear(d_model, d_model, bias=False)
        self.w_o = nn.Linear(d_model, d_model, bias=False)

    def forward(self, query, key, value, mask=None, return_attn=False):
        """query: (B, Tq, d), key/value: (B, Tk, d), mask: broadcastable bool."""
        B, Tq = query.shape[:2]
        Tk    = key.shape[1]

        q = self.w_q(query).view(B, Tq, self.n_heads, self.d_head).transpose(1, 2)
        k = self.w_k(key  ).view(B, Tk, self.n_heads, self.d_head).transpose(1, 2)
        v = self.w_v(value).view(B, Tk, self.n_heads, self.d_head).transpose(1, 2)

        if mask is not None and mask.ndim == 2:
            mask = mask.unsqueeze(0).unsqueeze(0)
        elif mask is not None and mask.ndim == 3:
            mask = mask.unsqueeze(1)

        dropout_p = self.dropout_p if self.training else 0.0
        head = F.scaled_dot_product_attention(q, k, v, mask, dropout_p=dropout_p)
        out  = head.permute(0, 2, 1, 3).reshape(B, Tq, self.d_model)
        out  = self.w_o(out)

        if return_attn:
            scale  = math.sqrt(self.d_head)
            scores = (q @ k.transpose(-2, -1)) / scale
            if mask is not None:
                scores = scores.masked_fill(mask == 0, float("-inf"))
            attn_w = F.softmax(scores, dim=-1)
            return out, attn_w

        return out


# adapted from NB09/NB13
class EncoderBlock(nn.Module):

    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.norm1     = nn.LayerNorm(d_model)
        self.norm2     = nn.LayerNorm(d_model)
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model), nn.Dropout(dropout)
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, src_mask=None):
        x = x + self.dropout(self.self_attn(self.norm1(x), self.norm1(x), self.norm1(x), mask=src_mask))
        x = x + self.ffn(self.norm2(x))
        return x

Defining the full `ViTEncoder`:

In [ ]:
class ViTEncoder(nn.Module):

    def __init__(self, img_size=224, patch_size=16, in_channels=3,
                 d_model=256, num_heads=8, num_layers=6, d_ff=512, dropout=0.1):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, d_model)
        self.dropout     = nn.Dropout(dropout)
        self.layers      = nn.ModuleList(
            [EncoderBlock(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)]
        )
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        x = self.dropout(self.patch_embed(x))  # (B, N+1, d_model)  # <1>
        for block in self.layers:
            x = block(x)
        x = self.norm(x)
        return x[:, 1:, :]                     # (B, N, d_model)     # <2>

1. Dropout applied to the patch embedding (standard ViT regularization).
2. We return all patch tokens (index 1 onwards), excluding `[CLS]`. These $N = 196$ tokens serve as the cross-attention memory for the decoder.

## Caption decoder

**Caption decoder.** The caption decoder is a stack of $M$ decoder blocks, each identical to the `DecoderBlock` from [NB13](./13-translation.html). Each block applies (1) causal self-attention over the caption prefix, (2) cross-attention over the $N = 196$ patch tokens produced by the ViT encoder, and (3) a position-wise FFN, all with Pre-LN residuals. At training time, **teacher forcing** is used: the full ground-truth caption (shifted right, prefixed with `<bos>`) is fed to the decoder in parallel. At inference, the caption is generated autoregressively token by token.

Re-implementing `DecoderBlock` for self-containedness (adapted from NB13):

In [ ]:
# adapted from NB13
class DecoderBlock(nn.Module):

    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.self_attn  = MultiHeadAttention(d_model, num_heads, dropout)  # <1>
        self.cross_attn = MultiHeadAttention(d_model, num_heads, dropout)  # <2>
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model), nn.Dropout(dropout)
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, tgt, memory, tgt_mask=None, return_attn=False):
        # sub-layer 1: causal self-attention over target tokens
        x = tgt + self.dropout(self.self_attn(self.norm1(tgt), self.norm1(tgt),
                                              self.norm1(tgt), mask=tgt_mask))  # <3>
        # sub-layer 2: cross-attention — queries from decoder, keys/values from encoder
        x2 = self.norm2(x)
        if return_attn:
            ca_out, attn_w = self.cross_attn(x2, memory, memory, return_attn=True)  # <4>
        else:
            ca_out = self.cross_attn(x2, memory, memory)
            attn_w = None
        x = x + self.dropout(ca_out)
        # sub-layer 3: position-wise FFN
        x = x + self.ffn(self.norm3(x))                                             # <5>
        if return_attn:
            return x, attn_w
        return x

1. Causal self-attention — the same `MultiHeadAttention` module, but receives a causal mask during the forward pass so each target position can only attend to earlier positions.
2. Cross-attention — `query` comes from the decoder stream, `key` and `value` come from the encoder's $N$ patch tokens. This is the mechanism that conditions caption generation on image content.
3. Pre-LN residual: normalize, apply self-attention, add back. The mask `tgt_mask` is a lower-triangular boolean matrix of shape $(T, T)$.
4. The encoder memory (patch tokens) is broadcast across all decoder layers — computed once and used at every decoding step.
5. The FFN processes each token embedding independently, adding non-linear capacity after cross-attention.

Defining the full `CaptioningModel` with positional encoding and mask helpers:

In [ ]:
def make_causal_mask(sz: int, device) -> torch.BoolTensor:
    """Lower-triangular boolean mask of shape (sz, sz)."""
    return torch.tril(torch.ones(sz, sz, dtype=torch.bool, device=device))


# adapted from NB09/NB13
class PositionalEncoding(nn.Module):

    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 5000):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe  = torch.zeros(max_len, d_model)
        t   = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        w   = torch.exp(-(torch.arange(0, d_model, 2).float() / d_model * math.log(10000)))
        pe[:, 0::2] = torch.sin(t * w)
        pe[:, 1::2] = torch.cos(t * w)
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1), :])


class CaptioningModel(nn.Module):

    def __init__(self, vocab_size, d_model=256, num_heads=8,
                 num_enc_layers=6, num_dec_layers=6, d_ff=512,
                 dropout=0.1, img_size=224, patch_size=16, max_len=64):
        super().__init__()
        self.d_model = d_model
        self.encoder = ViTEncoder(img_size, patch_size, 3, d_model, num_heads,
                                  num_enc_layers, d_ff, dropout)
        self.tgt_embed = nn.Embedding(vocab_size, d_model, padding_idx=PAD_IDX)
        self.pos_enc   = PositionalEncoding(d_model, dropout, max_len)
        self.decoder   = nn.ModuleList(
            [DecoderBlock(d_model, num_heads, d_ff, dropout) for _ in range(num_dec_layers)]
        )
        self.norm_dec = nn.LayerNorm(d_model)
        self.fc_out   = nn.Linear(d_model, vocab_size)
        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def encode(self, imgs):
        return self.encoder(imgs)  # (B, N, d_model)

    def decode(self, tgt, memory, tgt_mask, return_attn=False):
        x = self.pos_enc(self.tgt_embed(tgt) * math.sqrt(self.d_model))
        last_attn = None
        for i, block in enumerate(self.decoder):
            is_last = (i == len(self.decoder) - 1)
            if return_attn and is_last:
                x, last_attn = block(x, memory, tgt_mask=tgt_mask, return_attn=True)
            else:
                x = block(x, memory, tgt_mask=tgt_mask)
        x = self.norm_dec(x)
        if return_attn:
            return x, last_attn
        return x

    def forward(self, imgs, tgt, return_attn=False):
        memory  = self.encode(imgs)          # (B, N, d_model)
        T       = tgt.size(1)
        tgt_mask = make_causal_mask(T, imgs.device)
        if return_attn:
            dec_out, attn = self.decode(tgt, memory, tgt_mask, return_attn=True)
            return self.fc_out(dec_out), attn
        dec_out = self.decode(tgt, memory, tgt_mask)
        return self.fc_out(dec_out)

Printing the parameter count and verifying output shapes:

In [ ]:
torch.manual_seed(RANDOM_SEED)
_test_model = CaptioningModel(
    vocab_size=VOCAB_SIZE, d_model=256, num_heads=8,
    num_enc_layers=4, num_dec_layers=4, d_ff=512,
    dropout=0.1, img_size=224, patch_size=16, max_len=64,
).to(DEVICE)

n_params = sum(p.numel() for p in _test_model.parameters() if p.requires_grad)
print(f"Parameters: {n_params:,}")

# shape check
_dummy_img = torch.randn(2, 3, 224, 224, device=DEVICE)
_dummy_cap = torch.randint(0, VOCAB_SIZE, (2, 15), device=DEVICE)
_logits = _test_model(_dummy_img, _dummy_cap)
print(f"Logits shape: {_logits.shape}  (expected: 2, 15, {VOCAB_SIZE})")

_memory = _test_model.encode(_dummy_img)
print(f"Memory shape: {_memory.shape}  (expected: 2, 196, 256)")
del _test_model, _dummy_img, _dummy_cap, _logits, _memory

## Training

**Training setup.** We train with **teacher forcing**: at each step the decoder receives the ground-truth caption shifted right (prefixed with `<bos>`) and is trained to predict the next token. The loss is cross-entropy over the vocabulary, with padding positions excluded via `ignore_index=PAD_IDX`. The optimizer is AdamW with learning rate $3 \times 10^{-4}$ and cosine annealing over 20 epochs. Gradient clipping at 1.0 prevents exploding gradients early in training. The ViT encoder is trained from scratch — no pretrained weights — since the full pipeline is small enough to converge on Flickr8k within a reasonable time.

In [ ]:
#| output: false

NUM_EPOCHS     = 20
LR             = 3e-4
D_MODEL        = 256
NUM_HEADS      = 8
NUM_ENC_LAYERS = 4
NUM_DEC_LAYERS = 4
D_FF           = 512
DROPOUT        = 0.1
PATCH_SIZE     = 16
MAX_LEN        = 64

torch.manual_seed(RANDOM_SEED)
model = CaptioningModel(
    vocab_size=VOCAB_SIZE,
    d_model=D_MODEL, num_heads=NUM_HEADS,
    num_enc_layers=NUM_ENC_LAYERS, num_dec_layers=NUM_DEC_LAYERS,
    d_ff=D_FF, dropout=DROPOUT, img_size=IMG_SIZE,
    patch_size=PATCH_SIZE, max_len=MAX_LEN,
).to(DEVICE)

criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameters: {n_params:,}")

history = {"train_loss": [], "val_loss": []}


def run_epoch(model, loader, optimizer, criterion, device, train=True):
    model.train() if train else model.eval()
    total_loss = 0.0
    ctx = torch.enable_grad() if train else torch.inference_mode()
    with ctx:
        for imgs, captions in loader:
            imgs, captions = imgs.to(device), captions.to(device)
            tgt_in  = captions[:, :-1]                                          # <1>
            tgt_out = captions[:, 1:]                                           # <2>
            logits  = model(imgs, tgt_in)                 # (B, T-1, V)        # <3>
            loss    = criterion(
                logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1)        # <4>
            )
            if train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)              # <5>
                optimizer.step()
            total_loss += loss.item()
    return total_loss / len(loader)


for epoch in range(1, NUM_EPOCHS + 1):
    tr = run_epoch(model, train_loader, optimizer, criterion, DEVICE, train=True)
    va = run_epoch(model, val_loader,   None,      criterion, DEVICE, train=False)
    scheduler.step()
    history["train_loss"].append(tr)
    history["val_loss"].append(va)
    print(f"Epoch {epoch:02d} | train {tr:.4f} | val {va:.4f}")

1. `tgt_in` is the caption without its last token — fed to the decoder as input (teacher-forced).
2. `tgt_out` is the caption without its first (`<bos>`) token — the targets the decoder must predict.
3. The model returns logits of shape $(B, T-1, V)$ where $V$ is the vocabulary size.
4. Reshape to $(B(T-1), V)$ and $(B(T-1),)$ for `CrossEntropyLoss`.
5. Clip the global gradient norm to 1.0 to prevent early-training instability.

Plotting the training and validation loss curves:

In [ ]:
#| code-fold: true
#| label: fig-loss
#| fig-cap: "Training and validation cross-entropy loss over 20 epochs."

epochs = range(1, NUM_EPOCHS + 1)
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(epochs, history["train_loss"], color="C0", linewidth=2, label="train")
ax.plot(epochs, history["val_loss"],   color="C1", linewidth=2, label="val")
ax.set_xlabel("Epoch")
ax.set_ylabel("Cross-entropy loss")
ax.grid(linestyle="dotted", alpha=0.6)
ax.legend()
fig.tight_layout()
plt.show()

**Figure.** Training and validation cross-entropy loss over 20 epochs. Both curves decrease steadily and track closely, indicating that the model is learning without severe overfitting at this scale. The gap between train and validation loss widens slightly in later epochs — expected given that the encoder is trained from random initialization on a small dataset.

## Greedy decoding

**Greedy decoding.** At inference we encode the image once to obtain the $N = 196$ patch memory tokens, then generate the caption autoregressively. Starting from `<bos>`, at each step we append the argmax-predicted token, stopping at `<eos>` or `MAX_CAP_LEN` tokens. This is the simplest decoding strategy; beam search would improve quality at the cost of additional computation.

In [ ]:
@torch.inference_mode()
def generate_caption(model, img_tensor, max_len=MAX_CAP_LEN):
    model.eval()
    img       = img_tensor.unsqueeze(0).to(DEVICE)         # (1, 3, H, W)
    memory    = model.encode(img)                          # (1, N, d_model)
    dec_input = torch.tensor([[BOS_IDX]], device=DEVICE)
    for _ in range(max_len):
        T      = dec_input.size(1)
        causal = make_causal_mask(T, DEVICE)
        dec_out = model.decode(dec_input, memory, causal)
        logits  = model.fc_out(dec_out[:, -1, :])          # (1, V)
        next_id = logits.argmax(-1).item()
        dec_input = torch.cat(
            [dec_input, torch.tensor([[next_id]], device=DEVICE)], dim=1
        )
        if next_id == EOS_IDX:
            break
    ids = dec_input[0, 1:].tolist()
    return " ".join(itos[i] for i in ids if i not in (BOS_IDX, EOS_IDX, PAD_IDX))

Showing four test images with their generated captions:

In [ ]:
#| code-fold: true
#| label: fig-captions
#| fig-cap: "Four test images with greedy-decoded captions produced by the captioning model. The ground-truth caption is shown in gray below each generated caption."

fig, axes = plt.subplots(1, 4, figsize=(13, 4))
for i, ax in enumerate(axes):
    ex   = test_examples[i]
    pred = generate_caption(model, eval_transform(ex["image"].convert("RGB")))
    gt   = ex["caption"]

    ax.imshow(ex["image"])
    ax.axis("off")

    # wrap prediction
    pred_words = pred.split()
    pred_lines = []
    for j in range(0, len(pred_words), 7):
        pred_lines.append(" ".join(pred_words[j:j+7]))
    pred_str = "\n".join(pred_lines)

    gt_words = gt.split()
    gt_lines = []
    for j in range(0, min(len(gt_words), 14), 7):
        gt_lines.append(" ".join(gt_words[j:j+7]))
    gt_str = "gt: " + "\n    ".join(gt_lines)

    ax.set_title(pred_str + "\n" + gt_str, fontsize=7)

fig.tight_layout()
plt.show()

**Figure.** Test images with greedy-decoded captions (top line) and the corresponding ground-truth caption (gray, prefixed "gt:"). The model correctly identifies dominant subjects and actions in many cases. Errors typically involve hallucinated attributes or swapped object types — expected behavior from a small model trained from scratch on a few thousand images.

## BLEU score

**BLEU.** The BLEU-4 metric [@bleu] measures the geometric mean of 1- through 4-gram precision between a hypothesis and a reference, multiplied by a brevity penalty for short outputs. A corpus BLEU score averages over all test examples. We adapt the `ngram_counts`, `bleu4`, and `corpus_bleu` functions from [NB13](./13-translation.html).

In [ ]:
# adapted from NB13
def ngram_counts(tokens, n):
    return Counter(tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1))


def bleu4(hypotheses, references):
    """Corpus BLEU-4 (uniform weights, brevity penalty)."""
    clip_counts  = [0] * 4
    total_counts = [0] * 4
    hyp_len = ref_len = 0

    for hyp, ref in zip(hypotheses, references):
        hyp_toks = hyp.split()
        ref_toks = ref.split()
        hyp_len += len(hyp_toks)
        ref_len += len(ref_toks)
        for n in range(1, 5):
            hyp_ng = ngram_counts(hyp_toks, n)
            ref_ng = ngram_counts(ref_toks, n)
            for gram, cnt in hyp_ng.items():
                clip_counts[n-1]  += min(cnt, ref_ng.get(gram, 0))
                total_counts[n-1] += cnt

    p_n = [clip_counts[n] / max(total_counts[n], 1) for n in range(4)]
    if any(p == 0 for p in p_n):
        return 0.0

    log_avg = sum(math.log(p) for p in p_n) / 4
    bp = min(1.0, math.exp(1 - ref_len / max(hyp_len, 1)))
    return bp * math.exp(log_avg) * 100


@torch.inference_mode()
def corpus_bleu(model, examples):
    hypotheses = [generate_caption(model, eval_transform(ex["image"].convert("RGB")))
                  for ex in examples]
    references = [ex["caption"] for ex in examples]
    return bleu4(hypotheses, references)


score = corpus_bleu(model, test_examples)
print(f"BLEU-4 on test set: {score:.2f}")

:::{.callout-note}

BLEU compares against a single reference caption per image, while Flickr8k has five. Evaluating against all five references (multi-reference BLEU) would yield a notably higher score and is the standard evaluation protocol. The single-reference score here underestimates true caption quality.

:::

## Cross-attention visualization

**Cross-attention.** We visualize which image patches the decoder attends to when generating each caption word. For a given test image we run the model with `return_attn=True` to extract the cross-attention weights from the last decoder layer. The weights have shape $(1, H_\text{heads}, T_\text{cap}, N_\text{patches})$. Averaging over heads and reshaping to $\sqrt{N} \times \sqrt{N} = 14 \times 14$ gives a spatial attention map over the image for each generated word. We upsample this map to the image size and overlay it as a semi-transparent heatmap.

Extracting per-step cross-attention weights during greedy decoding:

In [ ]:
@torch.inference_mode()
def generate_with_attention(model, img_tensor, max_len=MAX_CAP_LEN):
    """Greedy decode and collect per-step cross-attention from last decoder layer."""
    model.eval()
    img       = img_tensor.unsqueeze(0).to(DEVICE)
    memory    = model.encode(img)                          # (1, N, d_model)
    dec_input = torch.tensor([[BOS_IDX]], device=DEVICE)
    all_attn  = []                                         # <1>

    for _ in range(max_len):
        T      = dec_input.size(1)
        causal = make_causal_mask(T, DEVICE)
        dec_out, attn_w = model.decode(
            dec_input, memory, causal, return_attn=True
        )                                                  # attn_w: (1, H, T, N)  # <2>
        step_attn = attn_w[0, :, -1, :].mean(0).cpu()     # (N,)                  # <3>
        all_attn.append(step_attn)

        logits  = model.fc_out(dec_out[:, -1, :])
        next_id = logits.argmax(-1).item()
        dec_input = torch.cat(
            [dec_input, torch.tensor([[next_id]], device=DEVICE)], dim=1
        )
        if next_id == EOS_IDX:
            break

    ids   = dec_input[0, 1:].tolist()
    words = [itos[i] for i in ids if i not in (BOS_IDX, EOS_IDX, PAD_IDX)]
    attn_mat = torch.stack(all_attn[:-1]).numpy()          # (T_cap, N)            # <4>
    return words, attn_mat

1. `all_attn` collects the cross-attention distribution over the $N = 196$ patch tokens at every decoding step.
2. `attn_w` has shape $(1, H_\text{heads}, T, N)$ where $T$ grows at each step; we only need the last position.
3. Taking the last position's attention and averaging over heads gives a $(N,)$ vector — the spatial attention weights for this generation step.
4. We drop the attention collected for the `<eos>` step and stack the remaining steps into a $(T_\text{cap}, N)$ matrix.

Visualizing the attention maps for selected words overlaid on the image:

In [ ]:
#| code-fold: true
#| label: fig-attn-viz
#| fig-cap: "Per-word cross-attention maps from the last decoder layer, averaged over heads and overlaid on the original image. Brighter regions indicate higher attention weight at that generation step."

ex       = test_examples[0]
img_pil  = ex["image"].convert("RGB")
img_t    = eval_transform(img_pil)
words, attn_mat = generate_with_attention(model, img_t)  # (T_cap, 196)

# pick up to 6 content words to visualize
stop = {"a", "an", "the", "is", "in", "on", "of", "with", "and", "are", "to", "at"}
vis_indices = [i for i, w in enumerate(words) if w not in stop][:6]
n_show = len(vis_indices)

patch_grid = int(math.sqrt(attn_mat.shape[1]))  # 14
img_np     = np.array(img_pil.resize((IMG_SIZE, IMG_SIZE)))

fig, axes = plt.subplots(1, n_show, figsize=(2.4 * n_show, 2.8))
if n_show == 1:
    axes = [axes]

for ax, idx in zip(axes, vis_indices):
    weights = attn_mat[idx]                              # (196,)
    heat    = weights.reshape(patch_grid, patch_grid)    # (14, 14)
    heat    = (heat - heat.min()) / (heat.max() - heat.min() + 1e-8)

    # upsample to image size
    heat_up = np.array(
        Image.fromarray((heat * 255).astype(np.uint8)).resize(
            (IMG_SIZE, IMG_SIZE), resample=Image.BILINEAR
        )
    ) / 255.0

    ax.imshow(img_np)
    ax.imshow(heat_up, cmap="viridis", alpha=0.5, vmin=0, vmax=1)
    ax.set_title(f'"{words[idx]}"', fontsize=9)
    ax.axis("off")

fig.tight_layout()
plt.show()

print("Generated:", " ".join(words))
print("Reference:", ex["caption"])

**Figure.** Cross-attention maps for selected content words, overlaid on the input image. Each panel shows which $14 \times 14$ patch grid regions the last decoder layer attends to when generating that word. Attention is averaged over heads and bilinearly upsampled to the image resolution. Ideally, words like "dog" or "ball" would light up the corresponding objects in the image.

:::{.callout-note}

Attention maps from a ViT trained from scratch on a small dataset are coarser than those from large pretrained models like CLIP. Pretrained features would reveal sharper localization of objects and attributes, since the encoder would have already learned semantically meaningful patch representations.

:::

■